In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import random
import matplotlib.pyplot as plt
import tysserand
from tqdm import tqdm

In [ ]:
def get_zscore_matrix(net_df, patient_id, sample_id):
    submatrix = net_df.loc[
    net_df.index == f'patient-{patient_id}_ROI-{sample_id}',
    net_df.columns[net_df.columns.str.endswith('Z')]
        ]
    return submatrix.drop('assort Z', axis=1)

# --- 4. Calcul de la matrice de potentiels psi ---
def compute_psi(z_matrix, beta=1.0):
    A = z_matrix / np.max(np.abs(z_matrix))  # normalisation entre -1 et 1
    psi = np.exp(beta * A)
    return psi

def get_psi_value(psi, type1, type2, patient, sample):
    psi_row = psi.loc[f"patient-{patient}_ROI-{sample}"]
    key1 = f"{type1} - {type2} Z"
    key2 = f"{type2} - {type1} Z"
    if key1 in psi_row.index:
        return psi_row[key1]
    elif key2 in psi_row.index:
        return psi_row[key2]
    else:
        return 1.0

def gibbs_sampling(nodes_df, edges_df, psi_row, cell_types, patient, sample, n_iter=20):
    """
    Ré-échantillonne les phénotypes en fonction des interactions locales (Gibbs Sampling).

    Paramètres :
    - nodes_df : DataFrame avec colonnes ['CellID', 'Phenotypes', 'X_position', 'Y_position']
    - edges_df : DataFrame avec colonnes ['source', 'target']
    - psi_row : Series avec les interactions entre types (psi)
    - cell_types : liste des types cellulaires possibles
    - n_iter : nombre d’itérations de Gibbs sampling

    Retourne :
    - new_nodes_df : même que nodes_df mais avec nouveaux 'Phenotypes'
    - new_edges_df : inchangé
    """
    
    # Copie locale des phénotypes
    id_to_type = nodes_df.set_index("CellID")["Phenotypes"].to_dict()
    types = list(cell_types)

    for _ in tqdm(range(n_iter), desc="Gibbs sampling"):
        for cell_id in nodes_df["CellID"]:
            # Trouver les voisins à partir d'edges_df
            neighbors = edges_df.loc[
                (edges_df["source"] == cell_id) | (edges_df["target"] == cell_id)
            ]
            neighbor_ids = set(neighbors["source"].tolist() + neighbors["target"].tolist()) - {cell_id}
            neighbor_types = [id_to_type[nid] for nid in neighbor_ids if nid in id_to_type]
            
            # Calcul des probabilités pour chaque type possible
            scores = []
            for T in types:
                if neighbor_types:
                    prod = 1.0
                    for NT in neighbor_types:
                        prod *= get_psi_value(psi_row, T, NT, patient, sample)
                else:
                    prod = 1.0
                scores.append(prod)

            # Normalisation en distribution de probas
            scores = np.array(scores)
            probs = scores / scores.sum() if scores.sum() > 0 else np.ones_like(scores) / len(scores)
            
            # Échantillonnage
            new_type = np.random.choice(types, p=probs)
            id_to_type[cell_id] = new_type

    # Mise à jour du DataFrame des noeuds
    new_nodes_df = nodes_df.copy()
    new_nodes_df["Phenotypes"] = new_nodes_df["CellID"].map(id_to_type)

    return new_nodes_df, edges_df.copy()


In [152]:
random.seed(42)
np.random.seed(42)

In [153]:
nodes_df = pd.read_parquet("../output_data/IMC_networks_sample/nodes_patient-A_ROI-01.parquet")
edges_df = pd.read_parquet("../output_data/IMC_networks_sample/edges_patient-A_ROI-01.parquet")
net_df = pd.read_parquet("../output_data/assortativity/IMC_net_stat.parquet")

In [154]:
net_df

,# total,% B cells,% DC,% M1,% M2,% T-Cell CD3,% T-Cell CD4,% T-Cell CD8,% Treg cells,% fibroblasts,...,undetermined - T-Cell CD8 Z,undetermined - Treg cells Z,undetermined - fibroblasts Z,undetermined - mature DC Z,undetermined - monocytes Z,undetermined - myeloid cells Z,undetermined - neutrophils Z,undetermined - tissue structure (collagen) Z,undetermined - tumor cells Z,undetermined - undetermined Z
id,,,,,,,,,,,,,,,,,,,,,
patient-A_ROI-01,33617,0.020823,0.003272,0.136092,0.020882,0.000149,0.095904,0.063391,0.022697,0.037154,...,-2.474166,-4.448462,-1.304552,-8.421005,7.430892,-21.142261,-47.942284,11.542873,-17.367687,80.019044
patient-E_ROI-02,14206,0.007602,0.002534,0.013515,0.224483,0.000070,0.049979,0.200901,0.015064,0.055188,...,-21.841796,-0.162838,-6.251354,-2.843951,0.577030,-1.804382,-1.384367,0.276140,-10.231093,15.026933
patient-B_ROI-02bis,20308,0.008667,0.008765,0.025950,0.047420,0.001083,0.110942,0.082775,0.026000,0.040871,...,-15.753562,-8.069802,-8.238298,-19.100215,1.586248,-5.256245,-0.753014,-2.747891,-36.523009,62.329217
patient-A_ROI-02,28028,0.011952,0.002283,0.063615,0.035536,0.000178,0.120594,0.085557,0.036999,0.016269,...,-4.185485,-1.018948,7.575092,-11.049763,2.304074,-8.139545,-50.288844,12.464262,-38.287653,49.372107
patient-G_ROI-01,22142,0.000723,0.033917,0.040782,0.004245,0.000045,0.024569,0.035950,0.013955,0.012510,...,4.224798,6.579189,5.147845,3.154827,0.172844,-20.442346,-23.728374,-1.082221,-56.591854,53.155046
patient-F_ROI-02,20509,0.002389,0.012141,0.029548,0.032864,0.000098,0.031108,0.019211,0.005266,0.055634,...,3.468928,0.740383,5.226228,1.316298,-4.639673,1.443199,-0.440941,-9.229326,-48.426825,40.335740
patient-C_ROI-01,21391,0.016643,0.008976,0.007573,0.023889,NaN,0.100837,0.035950,0.020008,0.006405,...,-2.883614,1.139314,-0.266398,-5.847445,0.570165,9.314521,-3.031602,-11.958124,-46.230110,41.528611
patient-A_ROI-03,32786,0.007381,0.004118,0.047642,0.010279,NaN,0.070182,0.083755,0.025224,0.021656,...,-6.968896,-4.088739,6.479829,-3.388764,2.298198,-27.230468,-47.701223,1.394083,-42.906585,56.142904
patient-F_ROI-01,10866,0.002761,0.008375,0.054850,0.038653,0.001012,0.054114,0.019971,0.001657,0.121756,...,-3.331627,-1.755228,-4.799695,1.994078,-1.837217,1.374918,-1.371925,-9.371305,-4.616239,13.782120


In [155]:
cell_types = sorted(nodes_df['Phenotypes'].unique())
type_to_idx = {t: i for i, t in enumerate(cell_types)}
idx_to_type = {i: t for t, i in type_to_idx.items()}
n_types = len(cell_types)

In [156]:
patient = 'A'
sample = '01'

In [157]:
results = {}
try:
    z_matrix = get_zscore_matrix(net_df,patient, sample)
    psi = compute_psi(z_matrix, beta=1.0)
except Exception as e:
    print(f"Erreur chargement z-matrix pour ({patient}, {patient}): {e}")


In [158]:
psi

,B cells - B cells Z,DC - B cells Z,DC - DC Z,M1 - B cells Z,M1 - DC Z,M1 - M1 Z,M2 - B cells Z,M2 - DC Z,M2 - M1 Z,M2 - M2 Z,...,undetermined - T-Cell CD8 Z,undetermined - Treg cells Z,undetermined - fibroblasts Z,undetermined - mature DC Z,undetermined - monocytes Z,undetermined - myeloid cells Z,undetermined - neutrophils Z,undetermined - tissue structure (collagen) Z,undetermined - tumor cells Z,undetermined - undetermined Z
id,,,,,,,,,,,,,,,,,,,,,
patient-A_ROI-01,1.154638,1.030299,1.166478,1.008572,0.994524,1.274061,1.04073,1.000819,1.011041,1.125826,...,0.987748,0.97808,0.993521,0.958911,1.037718,0.900018,0.787517,1.059198,0.917105,1.489873


In [ ]:
new_nodes, new_edges = gibbs_sampling(nodes_df, edges_df, psi, cell_types, patient, sample, n_iter=10)
